In [1]:
!pip install requests

In [2]:
import pandas as pd
import requests

# =========================================================
# STEP 1: Fetch Historical Weather Data from Open-Meteo
# =========================================================
print("Fetching weather data for Pune district...")

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 18.5204,
    "longitude": 73.8567,
    "start_date": "2021-01-01",
    "end_date": "2026-02-28",
    "daily": ["temperature_2m_mean", "precipitation_sum"],
    "timezone": "Asia/Kolkata"
}

response = requests.get(url, params=params)
weather_json = response.json()

weather_df = pd.DataFrame({
    'arrival_date': pd.to_datetime(weather_json['daily']['time']),
    'temp_mean': weather_json['daily']['temperature_2m_mean'],
    'rainfall_daily': weather_json['daily']['precipitation_sum']
})

Fetching weather data for Pune district...


In [3]:
# =========================================================
# STEP 2: Weather Feature Engineering
# =========================================================
print("Calculating and lagging weather features...")
weather_df = weather_df.sort_values('arrival_date')

weather_df['temp_mean_lag15'] = weather_df['temp_mean'].shift(15)
weather_df['rainfall_lag15'] = weather_df['rainfall_daily'].shift(15)

weather_df['rainfall_7d_sum'] = weather_df['rainfall_lag15'].rolling(window=7).sum()
weather_df['rainfall_30d_sum'] = weather_df['rainfall_lag15'].rolling(window=30).sum()
weather_df['temp_7d_avg'] = weather_df['temp_mean_lag15'].rolling(window=7).mean()

weather_df = weather_df.drop(columns=['temp_mean', 'rainfall_daily'])
weather_df = weather_df.dropna()

Calculating and lagging weather features...


In [4]:
# =========================================================
# STEP 3: Merge Weather with Master Price Dataset
# =========================================================
print("Loading master price dataset and merging...")

master_df = pd.read_csv('../../data/tomato_cleaned_15day.csv')
master_df['arrival_date'] = pd.to_datetime(master_df['arrival_date'])

final_df = pd.merge(master_df, weather_df, on='arrival_date', how='left')

missing_weather = final_df['rainfall_7d_sum'].isna().sum()
total_rows = len(final_df)
missing_pct = missing_weather / total_rows * 100
print(f"Rows missing weather data after merge: {missing_weather} ({missing_pct:.1f}%)")
if missing_pct > 30:
    print("WARNING: >30% of rows missing weather data. Proceeding without weather features may be needed.")

Loading master price dataset and merging...
Rows missing weather data after merge: 0 (0.0%)


In [5]:
# =========================================================
# STEP 4: Final Cleanup and Save
# =========================================================
weather_cols = ['temp_mean_lag15', 'rainfall_lag15', 'rainfall_7d_sum', 'rainfall_30d_sum', 'temp_7d_avg']
final_df[weather_cols] = final_df[weather_cols].ffill()

final_csv_name = '../../data/tomato_with_temp_15day.csv'
final_df.to_csv(final_csv_name, index=False)

print(f"Success! Final dataset saved to '{final_csv_name}' with {len(final_df)} records.")
print("Features included:", weather_cols)

Success! Final dataset saved to '../../data/tomato_with_temp_15day.csv' with 16763 records.
Features included: ['temp_mean_lag15', 'rainfall_lag15', 'rainfall_7d_sum', 'rainfall_30d_sum', 'temp_7d_avg']
